In [11]:
#读取业务mysql,demo如下

In [3]:
import pymysql
import pandas as pd

def read_from_mysql(sql=None):
    assert sql is not None, "sql must be provided"
    # 数据库连接配置
    config = {
        'host': '10.0.0.9',      # 数据库地址
        'port': 3306,             # 端口
        'user': 'bigquery_user',           # 用户名
        'password': 'xdf#42sdfkjdfiyuISYFj76',   # 密码
        'database': 'niuniuh5_db',    # 数据库名
        'charset': 'utf8mb4',     # 字符集
        'cursorclass': pymysql.cursors.DictCursor
    }

    try:
        # 建立连接
        connection = pymysql.connect(**config)
        res=[]
        try:
            # 方法1：使用 cursor 执行 SQL
            with connection.cursor() as cursor:
                cursor.execute(sql)
                result = cursor.fetchall()
                #print("--- Cursor Result ---")
                for row in result:
                    #print(row)
                    res.append(row)
            return pd.DataFrame(res)
        finally:
            # 关闭连接
            connection.close()
            
    except Exception as e:
        print(f"Error: {e}")

In [30]:
# df=read_from_mysql(f"""
#  SELECT SUBSTR(createtime, 1, 10),ngametype,count(1) cnt
#             FROM table_dz_game_info
#             WHERE SUBSTR(createtime, 1, 10)>= '{dt}'
#               group by ngametype,SUBSTR(createtime, 1, 10)
# """)

In [5]:
sql_400=f"""
SELECT 
    nplayerid,
    400 AS ngametype,
    brobot,
    ROUND(SUM(player_pnl), 2) AS total_score
FROM (
    SELECT 
        pnl.nplayerid,
        u.brobot,
        pnl.player_pnl
    FROM (
        SELECT 
            t1.sztoken,
            t2.nplayerid,
            t1.create_dt,
            CASE 
                WHEN t2.reward > 0 THEN t2.reward - t1.entry_fee
                ELSE -t1.entry_fee 
            END AS player_pnl
        FROM (
            SELECT DISTINCT
                sztoken,
                szroomname,
                createtime,
                SUBSTR(createtime, 1, 10) AS create_dt,
                CAST(JSON_EXTRACT(szRuleJson, '$.joinFeeUcoin') AS UNSIGNED) / 1000000 AS entry_fee,
                CAST(JSON_EXTRACT(szRuleJson, '$.awards[0].award') AS UNSIGNED) / 1000000 AS award
            FROM table_dz_game_info
            WHERE SUBSTR(createtime, 1, 10) = '{dt}'
              AND ngametype = 400
        ) t1
        LEFT JOIN (
            SELECT 
                sztoken,
                nplayerid,
                reward / 1000000 AS reward
            FROM table_dz_match_result
            WHERE SUBSTR(tCreateTime, 1, 10) = '{dt}'
              AND ngametype = 400
        ) t2 ON t1.sztoken = t2.sztoken
    ) pnl
    LEFT JOIN table_user u ON pnl.nplayerid = u.nplayerid
    WHERE pnl.create_dt = '{dt}'
) final
GROUP BY nplayerid, brobot;
"""

In [6]:
read_from_mysql(sql_400)

,nplayerid,ngametype,brobot,total_score
0,162085,400,0,-0.10
1,207131,400,1,-0.10
2,212641,400,0,0.26
3,219830,400,0,-0.10


In [4]:
dt='2026-01-10'

In [39]:
sql_79=f"""
SELECT 
    nplayerid,
    ngametype,
    brobot,
    ROUND(SUM(player_pnl), 2) AS total_score
FROM (
    SELECT 
        pru.nplayerid,
        pru.ngametype,
        tu.brobot,
        pru.player_pnl
    FROM (
        SELECT 
            t1.sztoken,
            t1.nplayerid,
            t1.ngametype,
            SUBSTR(t1.Createtime, 1, 10) AS create_dt,
            ROUND(t1.nscore, 4) AS player_pnl
        FROM table_dz_game_score_detail t1
        LEFT JOIN (
            SELECT sztoken, UCoinRatio, ngametype
            FROM table_dz_game_info
            WHERE SUBSTR(CreateTime, 1, 10) = '{dt}'
              AND ngametype IN (79, 74)
            GROUP BY sztoken, UCoinRatio, ngametype
        ) t2 ON t1.sztoken = t2.sztoken
        WHERE t1.ngametype IN (79, 74)
          AND SUBSTR(t1.Createtime, 1, 10) = '{dt}'
        GROUP BY t1.sztoken, t1.nplayerid, t1.ngametype, SUBSTR(t1.Createtime, 1, 10)
    ) pru
    LEFT JOIN table_user tu
      ON pru.nplayerid = tu.nplayerid
    WHERE pru.create_dt = '{dt}'
) final
GROUP BY nplayerid, ngametype, brobot;
"""

In [40]:
df=read_from_mysql(sql_79)

In [10]:
sql450=f"""
SELECT 
    nplayerid,
    ngametype,
    brobot,
    ROUND(SUM(gameresult), 2) AS total_score
FROM (
    SELECT 
        ngametype, 
        sztoken, 
        nplayerid,
        brobot,
        gameresult / 1000 AS gameresult,
        SUBSTR(Createtime, 1, 10) AS create_dt
    FROM table_dz_game_gold_score_detail
    WHERE SUBSTR(Createtime, 1, 10) = '{dt}'
) AS gold_base
GROUP BY nplayerid, ngametype, brobot;
"""

In [11]:
df=read_from_mysql(sql450)

In [13]:
df

,nplayerid,ngametype,brobot,total_score
0,136156,450,0,-105875.00
1,136565,450,0,459346.00
2,138501,450,0,-102850.00
3,139066,450,0,18005003.00
4,143749,450,0,-693674.00
...,...,...,...,...
831,221195,450,0,-285825.00
832,221197,450,0,147000.00
833,221205,450,0,0.00
834,221207,450,0,-500.00
